# Regression using TA-Regression Library

This notebook demonstrates **TA-Regression** for the California Housing Price Prediction problem.

The previous notebook (`03_model_experimenting`) compared several Scikit-learn models and identified a tuned Random Forest as the best predictive model.

Here we use TA-Regression to explore interpretable regression approaches on the same housing problem.

The notebook focuses on:

- TA-Regression ElasticNet
- TA-Regression Linear Mixed Effects
- coefficient inspection
- model performance comparison
- comparison with the tuned Random Forest benchmark

The Bayesian/NumPyro examples from the original sales-price notebook are intentionally omitted because they are not required for this capstone and are not part of the stable environment used for this project.


## 1. Setup

The housing data has already been cleaned, split and feature-engineered by the preceding notebooks.

We therefore consume:

- `train/housing/features`
- `train/housing/target`
- `test/housing/features`
- `test/housing/target`

No duplicate feature engineering is performed here.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import os.path as op
import warnings

import numpy as np
import pandas as pd
import joblib
from matplotlib import pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)


In [ ]:
from ta_lib.core.api import (
    create_context,
    get_dataframe,
    load_dataset,
    initialize_environment,
    DEFAULT_ARTIFACTS_PATH,
)
from ta_lib.regression.api import RegressionComparison, RegressionReport

from taregression.elastic_net_api import ElasticNet
from taregression.linear_mixed_effects_api import LinearMixedEffects

initialize_environment(debug=False, hide_warnings=True)

config_path = op.join("conf", "config.yml")
context = create_context(config_path)

artifacts_folder = DEFAULT_ARTIFACTS_PATH


## 2. Load the Housing Training and Test Data


In [ ]:
train_X = load_dataset(context, "train/housing/features")
train_y = load_dataset(context, "train/housing/target")
test_X = load_dataset(context, "test/housing/features")
test_y = load_dataset(context, "test/housing/target")

train_df = train_X.copy()
train_df["median_house_value"] = train_y["median_house_value"].values

test_df = test_X.copy()
test_df["median_house_value"] = test_y["median_house_value"].values

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)
display(train_df.head())


### Target and feature overview

The target is `median_house_value`.

`ocean_proximity` is retained as a categorical feature and also provides a natural grouping factor for the Linear Mixed Effects example.


In [ ]:
print("Target:", "median_house_value")
print("\nCategorical columns:")
print(train_X.select_dtypes(include=["object", "string", "category"]).columns.tolist())

print("\nNumeric columns:")
print(train_X.select_dtypes(include=["number"]).columns.tolist())


In [ ]:
# TA-Regression does not perform sklearn-style imputation.
# Fill missing values using training-set statistics.
train_df = train_df.copy()
test_df = test_df.copy()

bedrooms_median = train_df["total_bedrooms"].median()

train_df["total_bedrooms"] = train_df["total_bedrooms"].fillna(
    bedrooms_median
)
test_df["total_bedrooms"] = test_df["total_bedrooms"].fillna(
    bedrooms_median
)

train_df["bedrooms_per_room"] = (
    train_df["total_bedrooms"] / train_df["total_rooms"]
)
test_df["bedrooms_per_room"] = (
    test_df["total_bedrooms"] / test_df["total_rooms"]
)

## 3. TA-Regression ElasticNet

ElasticNet provides a regularized linear model and is useful here as an interpretable baseline.

The TA-Regression API accepts a Wilkinson-style model formula. Categorical `ocean_proximity` levels are represented using `C(ocean_proximity)`.


In [ ]:
numeric_features = [
    "longitude",
    "latitude",
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "median_income",
    "rooms_per_household",
    "bedrooms_per_room",
    "population_per_household",
]

elastic_formula = (
    "median_house_value ~ "
    + " + ".join(numeric_features)
    + " + C(ocean_proximity)"
)

print(elastic_formula)


In [ ]:
elastic_model = ElasticNet(
    elastic_formula,
    backend="sklearn",
)

elastic_model.fit(
    train_df,
    alpha=0.5,
    lmbd=0.003,
    max_iter=10000,
)

elastic_coefficients = elastic_model.get_coefficients()
elastic_coefficients


### ElasticNet predictions and report


In [ ]:
elastic_train_pred = np.asarray(elastic_model.predict(train_df)).reshape(-1)
elastic_test_pred = np.asarray(elastic_model.predict(test_df)).reshape(-1)

elastic_report = RegressionReport(
    x_train=train_X,
    y_train=train_y,
    x_test=test_X,
    y_test=test_y,
    yhat_train=elastic_train_pred,
    yhat_test=elastic_test_pred,
)

elastic_report.get_report(
    include_shap=False,
    file_path="reports/ta_reg_elastic_net_report",
)


In [ ]:
rf_pipeline = joblib.load("../../artifacts/housing_final_model.joblib")
rf_test_pred = np.asarray(
    rf_pipeline.predict(test_X)
).reshape(-1)

elastic_comparison = RegressionComparison(
    y=test_y["median_house_value"],
    yhats={
        "TA-Regression ElasticNet": elastic_test_pred,
        "Random Forest": rf_test_pred,
    },
)

elastic_metrics = elastic_comparison.perf_metrics()
display(elastic_metrics)

### ElasticNet coefficient visualization

The coefficients provide an interpretable view of how the housing variables contribute to predicted house value.


In [ ]:
if isinstance(elastic_coefficients, dict):
    coefficient_items = elastic_coefficients.items()
    coefficient_df = pd.DataFrame(
        coefficient_items,
        columns=["feature", "coefficient"],
    )
else:
    coefficient_df = pd.DataFrame(elastic_coefficients)

if {"feature", "coefficient"}.issubset(coefficient_df.columns):
    coefficient_plot = (
        coefficient_df.assign(abs_coefficient=lambda x: x["coefficient"].abs())
        .sort_values("abs_coefficient", ascending=False)
        .head(15)
        .sort_values("coefficient")
    )

    coefficient_plot.plot(
        x="feature",
        y="coefficient",
        kind="barh",
        figsize=(10, 6),
        legend=False,
        title="Top ElasticNet Coefficients",
    )
    plt.tight_layout()
    plt.show()
else:
    display(coefficient_df.head(20))


## 4. TA-Regression Linear Mixed Effects

Housing observations can be grouped by `ocean_proximity`.

A mixed-effects model allows the relationship between `median_income` and house value to vary by the `ocean_proximity` group while retaining common fixed effects.

Because mixed-effects models are substantially more expensive than the Random Forest benchmark, the experiment uses a reproducible subset of the training data.


In [ ]:
lme_columns = [
    "median_house_value",
    "longitude",
    "latitude",
    "housing_median_age",
    "median_income",
    "rooms_per_household",
    "bedrooms_per_room",
    "population_per_household",
    "ocean_proximity",
]

lme_train = train_df[lme_columns].dropna().copy()
lme_test = test_df[lme_columns].dropna().copy()

lme_sample = lme_train.sample(
    n=min(6000, len(lme_train)),
    random_state=context.random_seed,
)

print("LME training rows:", len(lme_sample))
print("LME test rows:", len(lme_test))


In [ ]:
lme_formula = (
    "median_house_value ~ "
    "longitude + latitude + housing_median_age + median_income + "
    "rooms_per_household + bedrooms_per_room + population_per_household + "
    "(median_income|ocean_proximity)"
)

print(lme_formula)


In [ ]:
lme_model = LinearMixedEffects(
    lme_formula,
    backend="statsmodels",
)

lme_model.fit(
    lme_sample,
    verbose=True,
)


In [ ]:
lme_coefficients = lme_model.get_coefficients()

fixed_effects = pd.DataFrame(
    lme_coefficients["common"].items(),
    columns=["feature", "coefficient"],
)

display(fixed_effects)


In [ ]:
lme_train_pred = np.asarray(lme_model.predict(lme_sample)).reshape(-1)
lme_test_pred = np.asarray(lme_model.predict(lme_test)).reshape(-1)

lme_report = RegressionReport(
    x_train=lme_sample.drop(columns=["median_house_value"]),
    y_train=lme_sample[["median_house_value"]],
    x_test=lme_test.drop(columns=["median_house_value"]),
    y_test=lme_test[["median_house_value"]],
    yhat_train=lme_train_pred,
    yhat_test=lme_test_pred,
)

lme_report.get_report(
    include_shap=False,
    file_path="reports/ta_reg_lme_report",
)


## 5. Model Comparison

The tuned Random Forest from Notebook 03 is used as the predictive benchmark.

The benchmark was:

- Random Forest
- `max_depth=20`
- `max_features=0.5`
- `min_samples_leaf=2`
- `n_estimators=215`
- `random_state=0`

Its Notebook 03 test performance was approximately:

- MAE: 31,727
- RMSE: 47,863
- R²: 0.827


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rf_test_path = op.join(
    artifacts_folder,
    "housing_final_model.joblib",
)

try:
    import joblib

    rf_model = joblib.load(rf_test_path)

    rf_test_pred = rf_model.predict(test_X)

    benchmark_metrics = {
        "Model": "Tuned Random Forest",
        "MAE": mean_absolute_error(test_y["median_house_value"], rf_test_pred),
        "RMSE": mean_squared_error(
            test_y["median_house_value"],
            rf_test_pred,
        ),
        "R2": r2_score(test_y["median_house_value"], rf_test_pred),
    }

    display(pd.DataFrame([benchmark_metrics]))

except FileNotFoundError:
    print(
        "Notebook 03 Random Forest artifact was not found at:",
        rf_test_path,
    )


In [ ]:
comparison_rows = [
    {
        "Model": "TA-Regression ElasticNet",
        "MAE": mean_absolute_error(
            test_y["median_house_value"],
            elastic_test_pred,
        ),
        "RMSE": mean_squared_error(
            test_y["median_house_value"],
            elastic_test_pred,
        ),
        "R2": r2_score(
            test_y["median_house_value"],
            elastic_test_pred,
        ),
    },
    {
        "Model": "TA-Regression LinearMixedEffects",
        "MAE": mean_absolute_error(
            lme_test["median_house_value"],
            lme_test_pred,
        ),
        "RMSE": mean_squared_error(
            lme_test["median_house_value"],
            lme_test_pred,
        ),
        "R2": r2_score(
            lme_test["median_house_value"],
            lme_test_pred,
        ),
    },
]

if "benchmark_metrics" in globals():
    comparison_rows.append(benchmark_metrics)

comparison_df = pd.DataFrame(comparison_rows).sort_values("RMSE")
display(comparison_df)


In [ ]:
comparison_plot = comparison_df.set_index("Model")[["RMSE"]]
comparison_plot.plot(
    kind="bar",
    figsize=(9, 5),
    title="Housing Model RMSE Comparison",
    ylabel="RMSE",
    legend=False,
)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


## 6. Conclusions

The TA-Regression experiments demonstrate two useful modelling approaches for the housing problem:

1. **ElasticNet** provides a regularized and interpretable linear baseline.
2. **Linear Mixed Effects** demonstrates how grouped housing characteristics can be modelled with group-specific effects.

The tuned Random Forest from Notebook 03 remains the primary predictive model because it provides the strongest predictive performance on this dataset.

TA-Regression is therefore complementary to the final production model: it provides interpretable regression and mixed-effects experimentation, while the Random Forest is retained for production scoring.


## 7. Reproducibility

All experiments use the project configuration and `context.random_seed`.

The notebook consumes the datasets generated by Notebook 02 and does not independently recreate the production feature-engineering pipeline.

Generated reports are stored under:

`notebooks/reference/reports/`
